In [1]:
# Import necessary packages
import torch
from torch import nn
import torchinfo
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import time


import brevitas.nn as qnn
from brevitas.quant import Int8Bias
from brevitas.inject.enum import ScalingImplType
from brevitas.inject.defaults import Int8ActPerTensorFloatMinMaxInit

# Prepare data loader
from torch.utils.data import Dataset, DataLoader
import h5py
from sklearn.metrics import accuracy_score

In [2]:
# Make sure that CUDA is available - without cuda we would be attempting to run the model on CPU which is tooooo slow. 
assert torch.cuda.is_available(), 'Cuda not available'

In [3]:
class radio_dataset(Dataset):
    all_IQ=[]
    all_mod=[]
    all_snr=[]
    train_sampler:torch.utils.data.SubsetRandomSampler
    val_sampler:torch.utils.data.SubsetRandomSampler
    test_sampler:torch.utils.data.SubsetRandomSampler
    def __init__(self,dataset_path:str,fpmsc:int=4096):
        iq_key="all_IQ_8bit"
        mod_key="all_labels"
        snr_key="all_SNRs"

        
        
        np.random.seed(2021) #maintain deterministic randomness
        print(f"loading {dataset_path}")
        h5_file = h5py.File(dataset_path,'r')
        # print(f"All available keys: {list(h5_file.keys())}")
        self.all_IQ = h5_file[iq_key]
        self.all_mod = h5_file[mod_key][:,0].flatten()
        self.all_snr = h5_file[snr_key][:,0].flatten()
        # print(f"Raw values shape: {self.all_IQ.shape}")
        # print(f"Raw values min/max range: {np.min(self.all_IQ)}, {np.max(self.all_IQ)}, {self.all_IQ.dtype}")
        # print(f"Labels shape: {self.all_mod.shape}")
        # print(f"SNR shape: {self.all_snr.shape}")
        
        train_indices = []
        test_indices = []
        val_indices = []
        # fpmsc: frames per mod snr combination, the dataset is organized as mod major, snr minor
        # each mod-snr-combination has fpsmc frames, we split those fpsmc frames uniformly to train, test, val indices
        for i in range(0,len(self.all_IQ),fpmsc):
            indices_subclass=list(range(i,i+fpmsc))
            split = int(np.ceil(0.8 * fpmsc)) #split 80,10,10
            split2 = int(np.ceil(0.9 * fpmsc))
            
            np.random.shuffle(indices_subclass)
            
            train_indices_subclass = indices_subclass[:split]
            val_indices_subclass = indices_subclass[split:split2]
            test_indices_subclass = indices_subclass[split2:]
            
            train_indices.extend(train_indices_subclass)
            test_indices.extend(test_indices_subclass)
            val_indices.extend(val_indices_subclass) 
        
        self.train_sampler = torch.utils.data.SubsetRandomSampler(train_indices)
        self.val_sampler = torch.utils.data.SubsetRandomSampler(val_indices)
        self.test_sampler = torch.utils.data.SubsetRandomSampler(test_indices)
        
        # print('Training set size: ',len(self.train_sampler))
        # print('Val set size: ',len(self.val_sampler))
        # print('Test set size: ',len(self.test_sampler))
    
    def __getitem__(self, idx):
        # transpose frame into Pytorch channels-first format (NCL = -1,2,1024)
        return self.all_IQ[idx].transpose(), self.all_mod[idx], self.all_snr[idx]

class rw_radio_dataset(Dataset):
    all_IQ=[]
    all_mod=[]
    all_snr=[]
    train_sampler:torch.utils.data.SubsetRandomSampler
    val_sampler:torch.utils.data.SubsetRandomSampler
    test_sampler:torch.utils.data.SubsetRandomSampler
    def __init__(self,dataset_path:str,fpmsc:int=4096):
        iq_key="all_IQ_8bit"
        mod_key="all_labels"
        snr_key="all_SNRs"

        
        np.random.seed(2021) #maintain deterministic randomness
        print(f"loading {dataset_path}")
        h5_file = h5py.File(dataset_path,'r')
        # print(f"All available keys: {list(h5_file.keys())}")
        self.all_IQ = h5_file[iq_key]
        self.all_mod = h5_file[mod_key][:].flatten()
        self.all_snr = np.full(self.all_mod.shape, -9999, dtype=np.float32)
        # print(f"Raw values shape: {self.all_IQ.shape}")
        # print(f"Raw values min/max range: {np.min(self.all_IQ)}, {np.max(self.all_IQ)}, {self.all_IQ.dtype}")
        # print(f"Labels shape: {self.all_mod.shape}")
        # print(f"SNR shape: {self.all_snr.shape}")
        
        train_indices = []
        test_indices = []
        val_indices = []
        # fpmsc: frames per mod snr combination, the dataset is organized as mod major, snr minor
        # each mod-snr-combination has fpsmc frames, we split those fpsmc frames uniformly to train, test, val indices
        for i in range(0,len(self.all_IQ),fpmsc):
            indices_subclass=list(range(i,i+fpmsc))
            split = int(np.ceil(0.8 * fpmsc)) #split 80,10,10
            split2 = int(np.ceil(0.9 * fpmsc))
            
            np.random.shuffle(indices_subclass)
            
            train_indices_subclass = indices_subclass[:split]
            val_indices_subclass = indices_subclass[split:split2]
            test_indices_subclass = indices_subclass[split2:]
            
            train_indices.extend(train_indices_subclass)
            test_indices.extend(test_indices_subclass)
            val_indices.extend(val_indices_subclass) 

        #we are testing on all IQ for now
        test_indices=np.arange(len(self.all_IQ))
        
        self.train_sampler = torch.utils.data.SubsetRandomSampler(train_indices)
        self.val_sampler = torch.utils.data.SubsetRandomSampler(val_indices)
        self.test_sampler = torch.utils.data.SubsetRandomSampler(test_indices)
        
        # print('Training set size: ',len(self.train_sampler))
        # print('Val set size: ',len(self.val_sampler))
        # print('Test set size: ',len(self.test_sampler))
    
    def __getitem__(self, idx):
        # transpose frame into Pytorch channels-first format (NCL = -1,2,1024)
        return self.all_IQ[idx].transpose(), self.all_mod[idx], self.all_snr[idx]

In [4]:
def quantized_vgg10(model_body_bit:int)->nn.Sequential:
    a_bits = model_body_bit  # a_bits is the bit width for ReLu
    w_bits = model_body_bit # w_bits is the bit width for all the weights
    
    input_bits:int=8
    new_min=-128.0
    new_max=128
    filters_conv = 64
    filters_dense = 128
    
    class InputQuantizer(Int8ActPerTensorFloatMinMaxInit):
        #Quantize to input_bits
        bit_width = input_bits
        
        #Min max value of the input. Set to the range value of the input 
        min_val = new_min #the min value of the input(dataset) before going through the model
        max_val = new_max #the max value of the input(dataset) before going through the model
        scaling_impl_type = ScalingImplType.CONST # Fix the quantization range to [min_val, max_val]
    
    return nn.Sequential(
            # Input quantization layer
            qnn.QuantHardTanh(act_quant=InputQuantizer),
        
            qnn.QuantConv1d(2, filters_conv, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(filters_conv, filters_conv, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(filters_conv, filters_conv, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(filters_conv, filters_conv, 3, padding=1, weight_bit_width=w_bits,bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(filters_conv, filters_conv, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(filters_conv, filters_conv, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(filters_conv, filters_conv, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_conv),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
            
            nn.Flatten(),
        
            qnn.QuantLinear(filters_conv*8, filters_dense, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_dense),
            qnn.QuantReLU(bit_width=a_bits),
        
            qnn.QuantLinear(filters_dense, filters_dense, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(filters_dense),
            qnn.QuantReLU(bit_width=a_bits, return_quant_tensor=True),
        
            qnn.QuantLinear(filters_dense, 15, weight_bit_width=w_bits, bias=True, bias_quant=Int8Bias),
        )

def vanilla_vgg10()->nn.Sequential:
    filters_conv = 64
    filters_dense = 128
    class VGG10(nn.Module):
        def __init__(self):
            super().__init__()
            
            # Conv block 1
            self.conv1 = nn.Conv1d(2, filters_conv, 3, padding=1)
            self.bn1 = nn.BatchNorm1d(filters_conv)
            self.relu1 = nn.ReLU()
            self.pool1 = nn.MaxPool1d(2)
            
            # Conv block 2
            self.conv2 = nn.Conv1d(filters_conv, filters_conv, 3, padding=1)
            self.bn2 = nn.BatchNorm1d(filters_conv)
            self.relu2 = nn.ReLU()
            self.pool2 = nn.MaxPool1d(2)
            
            # Conv block 3
            self.conv3 = nn.Conv1d(filters_conv, filters_conv, 3, padding=1)
            self.bn3 = nn.BatchNorm1d(filters_conv)
            self.relu3 = nn.ReLU()
            self.pool3 = nn.MaxPool1d(2)
            
            # Conv block 4
            self.conv4 = nn.Conv1d(filters_conv, filters_conv, 3, padding=1)
            self.bn4 = nn.BatchNorm1d(filters_conv)
            self.relu4 = nn.ReLU()
            self.pool4 = nn.MaxPool1d(2)
            
            # Conv block 5
            self.conv5 = nn.Conv1d(filters_conv, filters_conv, 3, padding=1)
            self.bn5 = nn.BatchNorm1d(filters_conv)
            self.relu5 = nn.ReLU()
            self.pool5 = nn.MaxPool1d(2)
            
            # Conv block 6
            self.conv6 = nn.Conv1d(filters_conv, filters_conv, 3, padding=1)
            self.bn6 = nn.BatchNorm1d(filters_conv)
            self.relu6 = nn.ReLU()
            self.pool6 = nn.MaxPool1d(2)
            
            # Conv block 7
            self.conv7 = nn.Conv1d(filters_conv, filters_conv, 3, padding=1)
            self.bn7 = nn.BatchNorm1d(filters_conv)
            self.relu7 = nn.ReLU()
            self.pool7 = nn.MaxPool1d(2)
            
            self.flatten = nn.Flatten()
            
            # Dense block 1
            self.fc1 = nn.Linear(filters_conv*8, filters_dense)
            self.bn8 = nn.BatchNorm1d(filters_dense)
            self.relu8 = nn.ReLU()
            
            # Dense block 2
            self.fc2 = nn.Linear(filters_dense, filters_dense)
            self.bn9 = nn.BatchNorm1d(filters_dense)
            self.relu9 = nn.ReLU()
            
            # Output layer
            self.fc3 = nn.Linear(filters_dense, 15, bias=True)
    
        def forward(self, x):
            # Conv blocks
            x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
            x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
            x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
            x = self.pool4(self.relu4(self.bn4(self.conv4(x))))
            x = self.pool5(self.relu5(self.bn5(self.conv5(x))))
            x = self.pool6(self.relu6(self.bn6(self.conv6(x))))
            x = self.pool7(self.relu7(self.bn7(self.conv7(x))))
            
            x = self.flatten(x)
            
            # Dense blocks
            x = self.relu8(self.bn8(self.fc1(x)))
            x = self.relu9(self.bn9(self.fc2(x)))
            x = self.fc3(x)
            
            return x
    return VGG10()


# print(torchinfo.summary(model,input_size=(1,2,1024),depth=1));

In [5]:
from src.test_result import *
def get_dataset_path(r:str,ppm:int):
    match (r,ppm):
        case ("rician",0):
            return "datasets/MatGenData_ppm0_rician_int8_20260213.h5"
        case ("rician",20):
            return "datasets/MatGenData_ppm20_rician_int8_20250828.h5"
        case ("rayleigh",0):
            return "datasets/MatGenData_ppm0_rayleigh_int8_20250928.h5"
        case ("rayleigh",20):
            return "datasets/MatGenData_ppm20_rayleigh_int8_20260204.h5"
        case _:
            assert False,"invalid params"
            return

def get_model_path(r:str,ppm:int,bit:int):
    match (r,ppm,bit):
        case ("rician",0,32):
            return "archive/rician_matgen_ppm0_2026_feb13_report/vanilla/model_vanilla.pth"
        case ("rician",0,8):
            return "archive/rician_matgen_ppm0_2026_feb13_report/8bit/model_brevitas.pth"
        case ("rician",0,4):
            return "archive/rician_matgen_ppm0_2026_feb13_report/4bit/model_brevitas.pth"
        case ("rician",0,2):
            return "archive/rician_matgen_ppm0_2026_feb13_report/2bit/model_brevitas.pth"

        case ("rician",20,32):
            return "archive/rician_matgen_ppm20_2026_jan22_report/mat_gen_vanilla_10112025/model_vanilla.pth"
        case ("rician",20,8):
            return "archive/rician_matgen_ppm20_2026_jan22_report/mat_gen_8bit_10112025/model_brevitas.pth"
        case ("rician",20,4):
            return "archive/rician_matgen_ppm20_2026_jan22_report/mat_gen_4bit_01222026/model_brevitas.pth"
        case ("rician",20,2):
            return "archive/rician_matgen_ppm20_2026_jan22_report/mat_gen_2bit_01222026/model_brevitas.pth"

        case ("rayleigh",0,32):
            return "archive/rayleigh_matgen_0ppm_2026_jan28_report/vanilla/model_vanilla.pth"
        case ("rayleigh",0,8):
            return "archive/rayleigh_matgen_0ppm_2026_jan28_report/8bit/model_brevitas.pth"
        case ("rayleigh",0,4):
            return "archive/rayleigh_matgen_0ppm_2026_jan28_report/4bit/model_brevitas.pth"
        case ("rayleigh",0,2):
            return "archive/rayleigh_matgen_0ppm_2026_jan28_report/2bit/model_brevitas.pth"
            
        case ("rayleigh",20,32):
            return "archive/rayleigh_matgen_20ppm_2026_feb28_report/vanilla/model_vanilla.pth"
        case ("rayleigh",20,8):
            return "archive/rayleigh_matgen_20ppm_2026_feb28_report/8bit/model_brevitas.pth"
        case ("rayleigh",20,4):
            return "archive/rayleigh_matgen_20ppm_2026_feb28_report/4bit/model_brevitas.pth"
        case ("rayleigh",20,2):
            return "archive/rayleigh_matgen_20ppm_2026_feb28_report/2bit/model_brevitas.pth"

        case _:
            assert False,"invalid params"
            return

def get_model(r:str,ppm:int,bit:int):
    load_path=get_model_path(r,ppm,bit)
    model=nn.Sequential()
    if bit==32:
        model=vanilla_vgg10()
    else:
        model=quantized_vgg10(bit)
    
    model.load_state_dict(torch.load(load_path))
    model.to('cuda')
    return model

def get_model_testing_result(model, dataset, batch_size:int=1024, use_snr=True)->testing_result:   
    model.to('cuda')
    # print(torchinfo.summary(model,input_size=(1,2,1024),depth=1));
    
    test_dataset=DataLoader(dataset, batch_size=batch_size, sampler=dataset.test_sampler)
    y_pred = np.empty((0)) 
    y_exp = np.empty((0))
    y_snr = np.empty((0))
    model.eval()

    with torch.no_grad():
        for data in tqdm(test_dataset, desc="Batches"):
            inputs, target, snr = data
            inputs = inputs.to('cuda').float()
            output = model(inputs).cpu()
            #model raw output 15 mods prediction values, we can do the softmax here
            output = np.argmax(output,axis=1)
            y_pred = np.concatenate((y_pred,output))
            y_exp = np.concatenate((y_exp,target))
            y_snr = np.concatenate((y_snr,snr))

    return testing_result(y_pred,y_exp,y_snr,use_snr=use_snr)

## Main

In [6]:
mod_classes = ["BPSK", 
                "QPSK", 
                "8PSK",
                "16QAM",
                "32QAM", 
                "64QAM", 
                "128QAM", 
                "256QAM",
                "16APSK", 
                "32APSK", 
                "64APSK", 
                "128APSK",
                "FM", 
                "AM-DSB-SC", 
                "AM-SSB-SC"]

lbl_count:int=len(mod_classes)
snr_classes=np.arange(0.0,31.0,2.0)
plot_parent_dir="cross_testing_result"
Path(plot_parent_dir).mkdir(exist_ok=True)

       
def main():    
    fading_models=["rician","rayleigh"]
    ppm_list=[0,20]
    bit_list=[32,8,4,2]
    count=0
    # for test_r in ["rician","rayleigh"]:
    #     for test_ppm in [0,20]:
    #         dataset=radio_dataset(get_dataset_path(test_r,test_ppm))
    #         for train_r in ["rician","rayleigh"]:
    #             for train_ppm in [0,20]:
    #                 for train_b in [32]:
    #                     model=get_model(train_r,train_ppm,train_b)
    #                     result=get_model_testing_result(model,dataset)
                        
    #                     file_name=f"./result/GPU_trained[{train_b}b_{train_r[0:3]}_ppm{train_ppm}]_test[{test_r[0:3]}_ppm{test_ppm}].npz"
    #                     assert not os.path.isfile(file_name), f"file {file_name} exist, avoid override"
    #                     result.save_file(file_name)
    #                     print(f"saved to {file_name}")

    dataset=rw_radio_dataset("datasets/Post_Processing_RW_Capture_03_09_int8.h5")
    for train_r in ["rician","rayleigh"]:
        for train_ppm in [20]:
            for train_b in [32]:
                model=get_model(train_r,train_ppm,train_b)
                result=get_model_testing_result(model,dataset, use_snr=False)
                
                file_name=f"./result/4mods_only/GPU_trained[{train_b}b_{train_r[0:3]}_ppm{train_ppm}]_test[RW_Post].npz"
                assert not os.path.isfile(file_name), f"file {file_name} exist, avoid override"
                result.save_file(file_name)
                print(f"saved to {file_name}")
                
main()

loading datasets/Post_Processing_RW_Capture_03_09_int8.h5


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

saved to ./result/4mods_only/GPU_trained[32b_ric_ppm20]_test[RW_Post].npz


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

saved to ./result/4mods_only/GPU_trained[32b_ray_ppm20]_test[RW_Post].npz
